In [1]:
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    pipeline
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


c:\Users\AMIT\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
text_col = 'clean_text' if 'clean_text' in df.columns else 'text'
label_col = 'label' if 'label' in df.columns else 'category'

if df[label_col].min() < 0:
    df['label'] = df[label_col].map({-1: 0, 0: 1, 1: 2})
else:
    df['label'] = df[label_col].astype(int)

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print(f'Train: {len(train_df)} | Val: {len(val_df)}')


NameError: name 'df' is not defined

In [ ]:
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_encodings = tokenizer(train_df[text_col].tolist(), truncation=True, max_length=128)
val_encodings = tokenizer(val_df[text_col].tolist(), truncation=True, max_length=128)

class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = SentimentDataset(train_encodings, train_df['label'])
val_dataset = SentimentDataset(val_encodings, val_df['label'])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
id2label = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
label2id = {'Negative': 0, 'Neutral': 1, 'Positive': 2}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
).to(device)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}


In [ ]:
eval_key = 'eval_strategy' if hasattr(TrainingArguments, 'eval_strategy') else 'evaluation_strategy'

training_args = TrainingArguments(
    output_dir='./distilbert_output',
    num_train_epochs=2,
    per_device_train_batch_size=32 if torch.cuda.is_available() else 16,
    per_device_eval_batch_size=32 if torch.cuda.is_available() else 16,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy='epoch',
    **{eval_key: 'epoch'},
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()


In [ ]:
eval_results = trainer.evaluate()
print('Evaluation Results:', eval_results)


In [ ]:
model.save_pretrained('./saved_distilbert')
tokenizer.save_pretrained('./saved_distilbert')

classifier = pipeline('sentiment-analysis', model='./saved_distilbert', tokenizer='./saved_distilbert', device=0 if torch.cuda.is_available() else -1)

samples = [
    'Amazing product, absolutely love it!',
    'The package was delivered today.',
    'Terrible quality, broke after one day.'
]
for res in classifier(samples):
    print(res)
